In [ ]:
# Run login and dashboard cells for the application. The index report is completed. Run final cell to show comparison results

import os 
from getpass import getpass # Hides logic values

os.environ["Grazioso_User"] = getpass("DB username: ")
os.environ["Grazioso_Password"] = getpass("DB password: ")
os.environ["Grazioso_Auth"] = "admin"

In [ ]:
from dash import Dash # Setup should use Dash no Jupyterdash as this Dash version already supports the notebook

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, ctx # Lets the dashboard figure out what sorting control the user just changed
from animal_merge_sort import merge_sort
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
# JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter, Display_Fields

from math import ceil # ceil rounds up when calculating number of pages

###########################
# Data Manipulation / Model
###########################
# Read the database login from enviornment
# Keeps sensitive information out of code
username = os.environ.get("Grazioso_User")
password = os.environ.get("Grazioso_Password")

# Stop with an informative message if needed info is missing
if not username or not password:
    raise RuntimeError("Set Grazioso_User and Grazioso_Password before running.")

# Connect to database via CRUD Module by passing in protected login
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
#df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
# error="ignore' prevents crash if there is no _id column
#df.drop(columns=['_id'],inplace=True, errors="ignore")

Page_Size = 10 # The dashboard will request 10 animals at a time

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

#Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#Place the HTML image tag in the line below into the app.layout code according to your design
#Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    dcc.Store(id="rescue-filter", data="all"), # Remembers rescue category when sorting controls change
    dcc.Store(id="database-page", data = 1), # Remember what database page is displayed right now
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode())), href='https://www.snhu.edu', target='_blank')),
    html.Center(html.B(html.H2('By Nicholas Deniz'))),
    html.Hr(),
    html.Div(className='buttonRow',
             style={'display' : 'flex'},
                 children=[
                     html.Button(id='water-button', n_clicks=0, children='Water Rescue'), 
                     html.Button(id='mountain-button', n_clicks=0, children='Mountain or Wilderness Rescue'), 
                     html.Button(id='disaster-button', n_clicks=0, children='Disaster or Individual Tracking'), 
                     html.Button(id='reset-button', n_clicks=0, children='Reset'), 
                  ]
# Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
    ),
    html.Hr(),
    # Lets user choose field and direction for the merge sort
    html.Div([
        html.Label("Sort this page by"), # Sorting will be applied to loaded animals
        dcc.Dropdown(
            id="sort-field",
            options=[
                {
                    "label": "Age in weeks",
                    "value": "age_upon_outcome_in_weeks",
                },
                {
                    "label": "Name",
                    "value": "name",
                },
                {
                    "label": "Breed",
                    "value": "breed",
                },
            ],
            # Start the sort by age
            value="age_upon_outcome_in_weeks",
            clearable=False,
        ),
        html.Label("Order"),

        dcc.RadioItems(
            id="sort-direction",
            options=[
                {
                    "label": "Ascending",
                    "value": "ascending",
                },
                {
                    "label": "Descending",
                    "value": "descending",
                },
            ],
            # Start with the smallest value first
            value="ascending",
            labelStyle={
                "display": "inline-block",
                "marginRight": "10px",
            },
        ),
    ], style={"maxWidth": "300px"}),

    html.Hr(),

#Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 
dash_table.DataTable(
    id='datatable-id',
    # Use approved fields instead of full database DataFrame
    columns=[
        {"name": field, "id": field, "deletable": False, "selectable": True} for field in Display_Fields
    ],
    data=[], # The callback will load the first database page
    #Set up the features for your interactive data table to make it user-friendly for your client
    row_selectable='single',
    column_selectable="single",
    selected_rows=[], # Nothing selected till animals are loaded in
    sort_action='none', # Getting rid of Dash's built in sort
    filter_action='none',
    page_action='none',

    style_table={
        "overflowX": "auto",
    },

    style_header={
        "backgroundColor": "white",
        "color": "black",
    },

    style_data={
        "backgroundColor": "white",
        "color": "black",
    },
),  

# Let user request another page from MongoDB
html.Div([
    html.Button(
        "Previous page",
        id = "previous-page",
        n_clicks = 0,
        disabled = True, # first pages doesn't have a previous page
    ),

    html.Span(
        "Loading animal records...",
        id = "page-status",
        style = { "margin": "10px"}
    ),

    html.Button(
        "Next page",
        id = "next-page",
        n_clicks = 0,
        disabled = True, # Callback enables it when another page happens
    ),
]),

html.P("Sorting and the breed chart follow the current page"),

html.Br(),
html.Hr(),
                                
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',)
            ]
        )
], style = {
    "backgroundColor": "white",
    "color": "black",
    "padding": "25px",
})

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback([Output("datatable-id", "data"), 
               Output("datatable-id", "selected_rows"), 
               Output("rescue-filter", "data"), 
               Output("database-page", "data"), 
               Output("page-status", "children"), 
               Output("previous-page", "disabled"), 
                Output("next-page", "disabled"), ], 

              [Input("water-button", "n_clicks"), 
               Input("mountain-button", "n_clicks"), 
               Input("disaster-button", "n_clicks"), 
               Input("reset-button", "n_clicks"),
               Input("sort-field", "value"),
               Input("sort-direction", "value"), 
               Input("previous-page", "n_clicks"), 
               Input("next-page", "n_clicks"),], 

              [State("rescue-filter", "data"), 
               State("database-page", "data"),],)

def update_dashboard(
    _water_clicks,
    _mountain_clicks,
    _disaster_clicks,
    _reset_clicks,
    sort_field,
    sort_direction,
    _previous_clicks,
    __next_clicks,
    rescue_filter,
    page, 
):
    """Get matching animals and sort the already loaded in records"""

    # Figure out which dashboard control was recently changed
    changed_id = ctx.triggered_id

    # Only a rescure-button click will changesaved rescue category
    # Changing the sorting controls will keep the current rescure category
    if changed_id == "water-button":
        rescue_filter = "water"
        page = 1

    elif changed_id == "mountain-button":
        rescue_filter = "mountain"
        page = 1

    elif changed_id == "disaster-button":
        rescue_filter = "disaster"
        page = 1

    elif changed_id == "reset-button":
        rescue_filter = "all"
        page = 1

    # Empty MongoDB query will return all animal records
    query = {}

#Water rescue
    if rescue_filter == "water":
        query = {
            "animal_type": "Dog", 
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        }
        
    #Mountain or wilderness Resuce
    elif rescue_filter == "mountain":
        query = {
            "animal_type": "Dog", 
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        }
        
    #Disaster or Individual Tracking Resuce
    elif rescue_filter == "disaster":
        query = {
            "animal_type": "Dog", 
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
        }

    try:
        # Checks if category is remembered and the page before using them
        if rescue_filter not in ["all", "water", "mountain", "disaster"]:
            raise ValueError("Choose valid rescue category")

        if type(page) is not int or page < 1:
            raise ValueError("Page haas to be a whole number starting at 1")

        # Accept only the sorting choices displayed by the dashboard
        if sort_field not in ["age_upon_outcome_in_weeks", "name", "breed"]:
            raise ValueError("Choose age, name, or breed when sorting")

        if sort_direction not in ["ascending", "descending"]:
            raise ValueError("Choose ascending or descending order")

        # Change page when only a page button is clicked
        if changed_id == "previous-page":
            page = max(1, page - 1) # Don't go below page 1

        elif changed_id == "next-page":
            page = page + 1

        # This will replace the original db.read(query) call
        result = db.read_page(query, page, Page_Size, Display_Fields)

        # counts all matching animals
        total = result["total"]

        # Round up: 31 animals needs 4 pages as each page only holds 10 animals
        last_page = max(1, ceil(total / Page_Size))

        # If the records were taken out then the remembered page may not exist
        if page > last_page:
            page = last_page
            result = db.read_page(query, page, Page_Size, Display_Fields)
            total = result["total"]
            last_page = max(1, ceil(total / Page_Size))

        # Keep merge sort. Sorts aniamls loaded for the current page
        sorted_records = merge_sort(result["records"], sort_field, descending = (sort_direction == "descending"),)

    except (ValueError, RuntimeError) as error:
        # Get rid of old results and show explanation
        return [], [], "all", 1, str(error), True, True

    # Select first animal when the page has stuff
    selected_rows = [0] if sorted_records else []

    # Show page info or explain that no matches were found
    if total == 0:
        status = "Not matching animals were found"
    else:
        status = f"Page {page} of {last_page} | {total} matching animals"
    
    # Display sorted records and return values in the same order as the Outputs
    return sorted_records, selected_rows, rescue_filter, page, status, page <= 1, page >= last_page # Disable Previous on first page while disable Next on last page

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(Output('graph-id', "children"), [Input('datatable-id', "data")]) # get currently loaded page
def update_graphs(viewData):
    # add code for chart of your choice (e.g. pie chart) #
    if not viewData:
        return [html.P("There are no animals that match the fitlers")]
    
    graph_data = pd.DataFrame.from_records(viewData)

    return [dcc.Graph(figure = px.pie(graph_data, names='breed', title='Breeds on the current page'))]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(Output('datatable-id', 'style_data_conditional'),[Input('datatable-id', 'selected_columns')])
def update_styles(selected_columns):
    #If nothing is selected return nothing
    if selected_columns is None:
        return[]
    
    return [{
        'if': { 'column_id': i },
        'backgroundColor': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(Output('map-id', "children"), [Input('datatable-id', "data"), Input('datatable-id', "selected_rows")]) # Animals on the curent page and the selected row
def update_map(viewData, index):  
    if not viewData or not index:
        return []
    
    row = index[0] # Table allows one selected row

    # Ignore a selection that doesn't belong to the current page
    if type(row) is not int or not 0 <= row < len(viewData):
        return[]

    animal = viewData[row] # Gets the selected animals dictionary

    # Read coords by name 
    try:
        latitude = float(animal.get("location_lat"))
        longitude = float(animal.get("location_long"))
    except (TypeError, ValueError):
        return [html.P("This animal doesn't have valid map coordinates")]

    # Coords have to be within valid ranges
    if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
        return [html.P("This animal doesn't have valid map coordinates")]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, 
            children=[
                dl.TileLayer(id="base-layer-id"),
                # Marker with tool tip and popup
                # Column 13 and 14 define the grid-coordinates for the map
                # Column 4 defines the breed for the animal
                # Column 9 defines the name of the animal
                dl.Marker(
                    position=[latitude, longitude], 
                    children=[
                        dl.Tooltip(animal.get("breed") or "Unknown breed"),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(animal.get("name") or "Unkown name")
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run(jupyter_mode='inline', debug=False) 

In [ ]:
print("Auth database:", os.environ.get("Grazioso_Auth"))

db = AnimalShelter(
    os.environ["Grazioso_User"],
    os.environ["Grazioso_Password"],
)

print("Animal count:", db.collection.count_documents({}))

In [ ]:
import json # Save measurements to a file
from pprint import pprint # Display query plan
from statistics import median # Find the median of the three times
from CRUD_Python_Module import AnimalShelter, Display_Fields

# One compund index that has multiple fields
Rescue_Index = [
    ("animal_type", 1),
    ("breed", 1),
    ("sex_upon_outcome", 1),
    ("age_upon_outcome_in_weeks", 1),
]

# Give index a name that shows what it has
Rescue_Index_Name = "rescue_type_breed_sex_age"

# using same water-rescue query as the dashbaord
query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
}

# Request same fields on dashboard
projection = {"_id": 0}
for field in Display_Fields:
    projection[field] = 1

def measure_query():
    # Store execution times
    times = []

    # Using three runs
    for run_number in range(3):
            # Tell MongoDB to run query and how it got the records
            explanation = db.database.command(
                "explain", 
                {
                    "find": db.collection.name,
                    "filter": query,
                    "projection": projection,
                    "sort": {"_id": 1}, 
                    "skip": 0, 
                    "limit": Page_Size,
                    "maxTimeMS": 5000,
                },
                verbosity = "executionStats",
            )

            # Get measurements from MongoDB
            stats = explanation["executionStats"]
            times.append(stats["executionTimeMillis"])

    # Counts from final run. Median time from all three runs
    measurements = {
        "Animals returned": stats["nReturned"],
        "Documents examined": stats["totalDocsExamined"],
        "Index's examined": stats["totalKeysExamined"],
        "Median time in ms": median(times),
    }

    # Steps MongoDB took to execute query
    plan = explanation["queryPlanner"]["winningPlan"]

    return measurements, plan

# Makes sure the index is absent before taking the baseline and checks the fields as the same index may have a different name
for information in db.collection.index_information().values():
    if list(information["key"]) == Rescue_Index:
        raise ValueError("This index already exists. Do not call this before a measurement")

# Keep first page so indexing preserves the results can be verified
before_page = db.read_page(
     query,
     page = 1,
     page_size = Page_Size,
     fields = Display_Fields,
)

# Uses search with real matches to compare
if before_page["total"] == 0:
    raise ValueError("No animals match. Use mountain or disaster query before adding the index")

# Measure and displau query before adding the index
before, before_plan = measure_query()
print("Before")
print(before)
pprint(before_plan)

# Save baseline before creation of index
with open("index_before.json", "w", encoding = "utf-8") as file:
    json.dump(
        {
            "query": query,
            "page_size": Page_Size,
            "projection": projection,
            "measurements": before,
            "plan": before_plan,
            "matching_animals": before_page["total"],
        },
        file, indent = 2, default = str,
    )

In [ ]:
# Create compound index
db.collection.create_index(
    Rescue_Index,
    name = Rescue_Index_Name,
)

# Repeat same measurement with the index
after, after_plan = measure_query()

## Gets the same page to see if the results stayed the same
after_page = db.read_page(
    query, 
    page = 1,
    page_size = Page_Size,
    fields = Display_Fields,
)

# Index should improve search without the results being changed
assert before_page["total"] == after_page["total"], "Count changed"
assert before_page["records"] == after_page["records"], "Records changed"

print("Before")
print(before)

print("\nAfter")
print(after)

print("\nResults stayed the same")
pprint(after_plan)

# Save both measurements and query plans
with open ("index_comparison.json", "w", encoding = "utf-8") as file:
    json.dump(
            {
                "query": query,
                "projection": projection,
                "page_size": Page_Size,
                "sort": {"_id": 1},
                "matching_animals": before_page["total"],
                "before": before,
                "after": after,
                "before_plan": before_plan,
                "after_plan": after_plan,
            },
            file, indent = 2, default = str,
        )

In [3]:
import json

with open ("index_comparison.json", "r", encoding = "utf-8") as file:
    comparison = json.load(file)

print("Before")
print(comparison["before"])

print("\nAfter")
print(comparison["after"])

Before
{'Animals returned': 10, 'Documents examined': 4222, "Index's examined": 4222, 'Median time in ms': 4}

After
{'Animals returned': 10, 'Documents examined': 17, "Index's examined": 22, 'Median time in ms': 0}
